# Function Testing Notebook

Author: Pete King

This notebook tests custom functions developed in the various helper modules to verify proper operation.

In [1]:
#123456789012345678901234567890123456789012345678901234567890123456789012345678
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import altair as alt
import yfinance as yf

import data_prep as dp

DATA_FILENAME='etf_raw_data.csv'
ETF='SPY'

# Deactivate the max rows and columns limit for Altair
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## Import and inspect ETF price data

In [2]:
df = pd.read_csv(
    DATA_FILENAME,
    index_col='date',
    parse_dates=True
)
df

,BIL,BND,GLD,HYG,IEF,IWM,LQD,QQQ,SPY,TIP,...,XLB,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
date,,,,,,,,,,,,,,,,,,,,,
1993-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.241396,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.413818,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.465538,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.724163,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.827618,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-13,91.510002,73.550003,460.839996,79.199997,95.589996,246.152130,108.169998,593.719971,662.289978,110.709999,...,49.189999,57.700001,48.889999,164.649994,136.800003,84.739998,42.250000,46.959999,149.789993,110.860001
2026-03-16,91.510002,73.830002,460.429993,79.449997,96.019997,248.477997,108.690002,600.380005,669.030029,111.059998,...,49.400002,57.900002,49.299999,166.059998,138.779999,84.980003,42.580002,47.259998,151.009995,112.199997
2026-03-17,91.519997,73.980003,459.269989,79.809998,96.190002,250.050003,109.300003,603.309998,670.789978,111.449997,...,49.520000,58.509998,49.560001,166.500000,139.539993,84.699997,42.720001,47.130001,149.639999,113.180000


## Compute and display daily return

Here we test the ability of the log_return function to compute daily returns and inspect the results.

In [3]:
test_df = df[[ETF]]
etf_return = test_df['SPY'].rolling(2).apply(dp.log_return, raw=True)
test_df[ETF + '_return'] = etf_return.values
test_df

,SPY,SPY_return
date,,
1993-01-29,24.241396,NaN
1993-02-01,24.413818,0.007088
1993-02-02,24.465538,0.002116
1993-02-03,24.724163,0.010516
1993-02-04,24.827618,0.004176
...,...,...
2026-03-13,662.289978,-0.005676
2026-03-16,669.030029,0.010125
2026-03-17,670.789978,0.002627


In [4]:
chart = alt.Chart(test_df.dropna().reset_index()).mark_circle(size=10).encode(
    x='date:T',
    y=ETF + '_return:Q'
)
chart.properties(height=200, width=800)

alt.Chart(...)

## Discussion

From the chart we can see that the mean of daily returns appears to be nearly zero -- an empirical justification of the zero mean assumption for expected return (E\[R\]).

Since volatility for an asset is a measure of the deviation of returns from expected return, we can get a feel for an asset's volatility just by inspecting the plot.  We see a general trend of baseline low volatility (for example, from Jan 2004 to Jan 2007), with periods of high volatility that tend to gradually revert to baseline (for example during the 'Great Recession', from late 2007, spiking in late 2008 / early 2009, and gradually reverting to a lower baseline by roughly 2012).

In [5]:
etf_return.describe()

count    8340.000000
mean        0.000396
std         0.011722
min        -0.115887
25%        -0.004337
50%         0.000678
75%         0.005916
max         0.135577
Name: SPY, dtype: float64

The main idea with this project is to think of the daily return for each asset as a random variable (R), and then investigate its statistical properties.  

***Right away, from this simple statistical description (above), we can get an idea of what to expect for the properties of an asset's returns (R):***
 - Estimated **expected return** (E\[R\]): 0.04 percent (very close to zero)
 - Estimated long-term (baseline) **volatility**: 1.17 percent

*Note that the financial term "volatility" can have mean interpretations, but here we mean the long-term standard deviation of return (R), assuming zero mean.*

In [6]:
# Compute using the zero-mean assumption for expected return
vol = np.sqrt(
    np.sum(etf_return.dropna().values**2) / len(etf_return.dropna())
)
print(f'Estimated long-term volatility with zero-mean assumption: \
        {vol * 100:2.2f} percent'
     )

Estimated long-term volatility with zero-mean assumption:         1.17 percent
